# Building and Running the Fraud Detection Pipeline

In this notebook, we'll build and run a Kubeflow Pipeline to train our fraud detection model.

In [7]:
import sys
import os

# Add the project root directory to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))  # Adjust if needed
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added {project_root} to Python path")

In [8]:
# Import required libraries
import kfp
import pandas as pd
import os
import numpy as np

## Compile and Run the Pipeline

In [9]:
# Import our pipeline
from src.pipeline.pipeline import fraud_detection_pipeline

# Compile the pipeline
pipeline_func = fraud_detection_pipeline
pipeline_filename = "fraud_detection_pipeline.yaml"
kfp.compiler.Compiler().compile(pipeline_func, pipeline_filename)

print(f"Pipeline compiled to {pipeline_filename}")

Pipeline compiled to fraud_detection_pipeline.yaml


In [10]:
from src.client.client_manager import KFPClientManager

# initialize a KFPClientManager
kfp_client_manager = KFPClientManager(
    api_url="http://localhost:8080/pipeline",
    skip_tls_verify=True,

    dex_username="user@example.com",
    dex_password="12341234",

    # can be 'ldap' or 'local' depending on your Dex configuration
    dex_auth_type="local",
)

# get a newly authenticated KFP client
# TIP: long-lived sessions might need to get a new client when their session expires
kfp_client = kfp_client_manager.create_kfp_client()

# Create an experiment
experiment_name = "fraud-detection-kserve"
experiment = kfp_client.create_experiment(name=experiment_name, namespace="kubeflow-user-example-com")

# Submit the pipeline run
run = kfp_client.create_run_from_pipeline_package(
    experiment_id=experiment.experiment_id,
    run_name="fraud-detection-training",
    pipeline_file=pipeline_filename,
    arguments={
        "model_name": "fraud-detection",
        "model_version": "v11"
    }
)

print(f"Pipeline run submitted with ID: {run.run_id}")

/Users/prashanth.chaitanya/git-workspaces/kubeflow/kserve-example/.venv/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


Pipeline run submitted with ID: 4f0e5fcf-f5a6-4f21-a29c-d67bb885b717


## Monitor the Pipeline Run

In [11]:
# Get pipeline run status
run_details = kfp_client.get_run(run.run_id)
print(f"Pipeline status: {run_details.state}")

Pipeline status: RUNNING


In [12]:
# Get the auth cookie using the credentials from your setup
from src.client.client_manager import KFPClientManager
manager = KFPClientManager(
    api_url='http://localhost:8080/pipeline',
    skip_tls_verify=True,
    dex_username='user@example.com',
    dex_password='12341234',
    dex_auth_type='local'
)
cookies = manager._get_session_cookies()
print(cookies)



oauth2_proxy_kubeflow=yYKcwplmoAwucVQ4o6cl-q7r9qNjcz3FUyQWWkrJ6lXdf59tY70rv2QNwOed_urRdkbB4eoAHaEzLldCGP_Ny1RZfvqAXFq6T4fz4QanyZ_nJcSqv1tIufVzz92PVlhbns9WUqKOm5m-YlI-JLnI6-DEH_goDTODUGO-tNk7ib2Egu_uJK63WZ9ESd5UE7sDENsu1-UYr0CFDH6RwITGKq0MaqV6N6HX7sUy5PVN1wsmvz2BmVUwMKiMSlw0KqgcmHWUiQs9bLk8mpMEl8mFo3ZFMVFQEiK0GKEl6Up9M3qJnTanh_1ynk_0oTprIKA9NQ86b4wbhhfcBws6E5jgo1DtMN7on3NAURxw1M4TpagfKJB7mTgRmj71VpuVQWr3MhqSuqAWywMIEJ9HuI7Ov0lodVLbldrwr17YAElqqDBOJldWshbVnFtSX6xLynCZXjmJswxeX3f8-eBOjKBInJt-A1ky8_R24DZXtEDEj0STHTLTplnMjgY0txBezyu5BNrby5JSOQ0Q4Y8ogB3jyAbFn-tIrkQNrqX4FgM15pP2QuQP9CN8Dx-izHG2gXLsGNfNjigdcMnZYaGkRSOuWjGR3wQZS1YRFgLfgQN-7fs302uMH2N6le9Q3WlFejPOmihY18FuRFiRI7_vaHbgIAQMJ2GoEg8gSiMf1KhYIjAG2ch4F6UeVOBKkmNuO0eeeQ1F1sZGoRffzDD-1VVbDOmSw9ze_Dmb8YgMdLvugkkaU7aW9d2-2jFRRJw84aTcagGYhhtrnh67qIosdk6Zpx_yfpLgDTcDfUwrXVaQKl0glTS7-0Mn3XCvTMP7wlJEA0kYA6a_QgGu7j92MfNFB0zalGMs1AMjj4iFLJzmVjH27NQhyGyiLlM0Z1PfBHQsKu48OOjwsobkrRWEs0zF6w9wOh_DBF4xt1qTb8_fpbuaRONCSlPWALilVQGIh4e0sRgAdbQu2EUNuoPbwo